In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
from sklearn.impute import KNNImputer
import os 
import random
from scipy.interpolate import splrep, BSpline
import json

In [ ]:
def carregar_dados_json(caminho_arquivo):
    with open(caminho_arquivo, 'r') as arquivo:
        dados = json.load(arquivo)
    return dados

# Função para filtrar os dados dos modelos GRU
def filtrar_dados(dados, model):
    data = {key: value for key, value in dados.items() if model in key}
    return data

# Função para extrair RMSE, imputações, links e protocolos
def extrair_dados(data):
    data_list = []
    for key, value in data.items():
        rmse = value['RMSE']
        imputation = key.split(" - ")[0]  # Extrair método de imputação
        link = key.split(" ")[-3]  # Extrair link (es-ce, ce-sp, etc.)
        protocol = key.split(" ")[3]  # Extrair protocolo
        data_list.append({'Link': link, 'Imputation': imputation, 'Protocol': protocol, 'RMSE': rmse})
    return data_list

def plotar_dados(data_list, model):
    df = pd.DataFrame(data_list)
    # Combinar 'Link' e 'Protocol' para diferenciar melhor as entradas
    df['Link_Protocol'] = df['Link'] + ' - ' + df['Protocol']

    # Verificar duplicatas
    duplicates = df[df.duplicated(subset=['Link_Protocol', 'Imputation'], keep=False)]
    if not duplicates.empty:
        print("Duplicatas encontradas nas seguintes combinações de 'Link_Protocol' e 'Imputation':")
        print(duplicates[['Link_Protocol', 'Imputation', 'RMSE']])
        print("\nOs RMSEs serão agregados usando a média.")

    # Usar pivot_table com uma função de agregação
    pivot_df = df.pivot_table(index='Link_Protocol', columns='Imputation', values='RMSE', aggfunc='mean')

    pivot_df.plot(kind='bar', figsize=(12, 7))
    plt.xlabel('Link e Protocolo')
    plt.ylabel('RMSE')
    plt.title(f'RMSE dos Modelos {model} por Imputação, Link e Protocolo')
    plt.legend(title='Imputação', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

models = ['GRU', 'LSTM']

for model in models:
    caminho_arquivo =  'evaluation_rmse_mae_2.json' 
    dados = carregar_dados_json(caminho_arquivo)
    data = filtrar_dados(dados, model)
    data_list = extrair_dados(data)
    plotar_dados(data_list, model)

In [ ]:
def carregar_dados_json(caminho_arquivo):
    with open(caminho_arquivo, 'r') as arquivo:
        dados = json.load(arquivo)
    return dados

def extrair_nomes_links(dados):
    links = set()
    for key in dados.keys():
        # Supondo que o nome do link esteja em uma posição específica na chave
        # Ajuste o índice conforme a estrutura real das suas chaves
        partes_chave = key.split(" ")
        if len(partes_chave) > 3:
            link = partes_chave[-3]  # Extrair o nome do link
            links.add(link)
    return list(links)

# Uso das funções
caminho_arquivo = 'evaluation_rmse_mae_2.json'  # Substitua pelo caminho do seu arquivo JSON
dados = carregar_dados_json(caminho_arquivo)
nomes_links = extrair_nomes_links(dados)

print("Nomes dos links encontrados no JSON:")
for link in nomes_links:
    print(link)

In [ ]:
def calcular_porcentagem (df):
    cont = df['Throughput'].isna().sum()
    porcentagem = 100 - (((len(df) - cont) * 100) / len(df))
    return porcentagem

def verificar_porcentagem_arquivo(caminho_arquivo):
    if caminho_arquivo.endswith('.csv'):
        df = pd.read_csv(caminho_arquivo)   
        porcentagem = calcular_porcentagem(df)
        return porcentagem

caminho_bbr = "datasets/throughput/treated"
porcentagem_bbr = {}
protocols = ['bbr', 'cubic']
for protocol in protocols:
    nome1 = os.path.join(caminho_bbr, protocol)
    for arquivo in os.listdir(nome1):
        caminho_arquivo = os.path.join(nome1, arquivo)
        for link in nomes_links:
            if link in caminho_arquivo and caminho_arquivo.endswith('csv'):
                porcentagem_bbr[arquivo] = verificar_porcentagem_arquivo(caminho_arquivo)

porcentagem_bbr

In [ ]:
labels = list(porcentagem_bbr.keys())
sizes = list(porcentagem_bbr.values())

plt.figure(figsize=(10, 6))
plt.barh(labels, sizes, color='red')
plt.xlabel('Percentage (%)')
plt.title('Porcentagens de Falhas por Link')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Carregar os dados
df = pd.read_csv('datasets/throughput/treated/bbr/treated bbr esmond data ba-ap 07-08-2023.csv')
df_imputed = pd.read_csv('datasets/dados-vazao-imputados/svd/svd/treated bbr esmond data ba-ap 07-08-2023.csv')

# Identificar os índices com valores ausentes no df original
nan_indices = df['Throughput'].isna()

# Plotagem
plt.figure(figsize=(16, 6))
plt.plot(df_imputed['Timestamp'], df_imputed['Throughput'], linestyle='-', linewidth=0.8, color='teal', label='Interpolated Data')
plt.scatter(df.loc[nan_indices, 'Timestamp'], df_imputed.loc[nan_indices, 'Throughput'], color='red', label='Filled Points', s=8)
plt.title('Throughput with Linear Interpolation')
plt.xlabel('Timestamp')
plt.ylabel('Throughput')
plt.legend()
plt.show()


In [ ]:


# Carregar os dados
df = pd.read_csv('datasets/throughput/treated/bbr/treated bbr esmond data ba-ap 07-08-2023.csv')
df_imputed = pd.read_csv('datasets/dados-vazao-imputados/svd/svd/treated bbr esmond data ba-ap 07-08-2023.csv')

def plot_imputed(df, df_imputed):
    # Identificar os índices com valores ausentes no df original
    nan_indices = df['Throughput'].isna()

    # Plotagem
    plt.figure(figsize=(16, 6))
    plt.plot(df_imputed['Timestamp'], df_imputed['Throughput'], linestyle='-', linewidth=0.8, color='teal', label='Interpolated Data')
    plt.scatter(df.loc[nan_indices, 'Timestamp'], df_imputed.loc[nan_indices, 'Throughput'], color='red', label='Filled Points', s=8)
    plt.title('Throughput with Linear Interpolation')
    plt.xlabel('Timestamp')
    plt.ylabel('Throughput')
    plt.legend()
    plt.show()



In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

def compare_and_plot_datasets(path_original, path_imputed):
    import os
    import pandas as pd
    import matplotlib.pyplot as plt

    # Get list of CSV files in each directory
    original_files = [f for f in os.listdir(path_original) if f.endswith('.csv')]
    imputed_files = [f for f in os.listdir(path_imputed) if f.endswith('.csv')]

    # Create dictionaries to map link names to filenames
    original_files_dict = {}
    imputed_files_dict = {}

    # Function to extract link name from filename
    def extract_link_name(filename):
        # Split the filename by spaces
        parts = filename.replace('.csv', '').split()
        # Assuming the link name is always at index 4
        # Adjust the index if your filenames have a different structure
        try:
            link_name = parts[4]
        except IndexError:
            # Handle cases where the filename does not have enough parts
            link_name = None
        return link_name

    # Build the dictionaries for original files
    for f in original_files:
        link_name = extract_link_name(f)
        if link_name:
            original_files_dict[link_name] = f

    # Build the dictionaries for imputed files
    for f in imputed_files:
        link_name = extract_link_name(f)
        if link_name:
            imputed_files_dict[link_name] = f

    # Find matching link names
    matching_link_names = set(original_files_dict.keys()).intersection(imputed_files_dict.keys())

    if not matching_link_names:
        print("No matching datasets found based on link names.")
        return

    # Process each matched pair
    for link_name in matching_link_names:
        original_file = os.path.join(path_original, original_files_dict[link_name])
        imputed_file = os.path.join(path_imputed, imputed_files_dict[link_name])

        # Load datasets
        df = pd.read_csv(original_file)
        df_imputed = pd.read_csv(imputed_file)

        # Compare the lengths and truncate the larger dataset
        min_length = min(len(df), len(df_imputed))
        df = df.head(min_length)
        df_imputed = df_imputed.head(min_length)

        # Identify missing indices in the original dataset
        nan_indices = df['Throughput'].isna()

        # Plotting
        plt.figure(figsize=(16, 6))
        plt.plot(df_imputed['Timestamp'], df_imputed['Throughput'],
                 linestyle='-', linewidth=0.8, color='teal', label='Interpolated Data')
        plt.scatter(df.loc[nan_indices, 'Timestamp'], df_imputed.loc[nan_indices, 'Throughput'],
                    color='red', s=10, label='Filled Points')
        plt.title(f'Throughput with Imputation for link {link_name}')
        plt.xlabel('Timestamp')
        plt.ylabel('Throughput')
        plt.legend()
        plt.show()

        # Uncomment the following lines to save the plots as files
        # plot_filename = f'plot_{link_name}.png'
        # plt.savefig(plot_filename)
        # plt.close()

        print(f"Processed and plotted dataset for link: {link_name}")

# Example usage:
compare_and_plot_datasets('datasets/throughput/treated/bbr', 'datasets/dados-vazao-imputados/svd/svd')


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

def compare_and_plot_datasets(path_original, path_imputed):
    import os
    import pandas as pd
    import matplotlib.pyplot as plt

    # Get list of CSV files in each directory
    original_files = [f for f in os.listdir(path_original) if f.endswith('.csv')]
    imputed_files = [f for f in os.listdir(path_imputed) if f.endswith('.csv')]

    # Create dictionaries to map link names to filenames
    original_files_dict = {}
    imputed_files_dict = {}

    # Function to extract link name from filename
    def extract_link_name(filename):
        # Split the filename by spaces
        parts = filename.replace('.csv', '').split()
        # Assuming the link name is always at index 4
        # Adjust the index if your filenames have a different structure
        try:
            link_name = parts[4]
        except IndexError:
            # Handle cases where the filename does not have enough parts
            link_name = None
        return link_name

    # Build the dictionaries for original files
    for f in original_files:
        link_name = extract_link_name(f)
        if link_name:
            original_files_dict[link_name] = f

    # Build the dictionaries for imputed files
    for f in imputed_files:
        link_name = extract_link_name(f)
        if link_name:
            imputed_files_dict[link_name] = f

    # Find matching link names
    matching_link_names = set(original_files_dict.keys()).intersection(imputed_files_dict.keys())

    if not matching_link_names:
        print("No matching datasets found based on link names.")
        return

    # Process each matched pair
    for link_name in matching_link_names:
        original_file = os.path.join(path_original, original_files_dict[link_name])
        imputed_file = os.path.join(path_imputed, imputed_files_dict[link_name])

        # Load datasets
        df = pd.read_csv(original_file)
        df_imputed = pd.read_csv(imputed_file)

        # Compare the lengths and truncate the larger dataset
        min_length = min(len(df), len(df_imputed))
        df = df.head(min_length)
        df_imputed = df_imputed.head(min_length)

        # Identify missing indices in the original dataset
        nan_indices = df['Throughput'].isna()

        # Plotting
        plt.figure(figsize=(16, 6))
        plt.plot(df_imputed['Timestamp'], df_imputed['Throughput'],
                 linestyle='-', linewidth=0.8, color='teal', label='Interpolated Data')
        plt.scatter(df.loc[nan_indices, 'Timestamp'], df_imputed.loc[nan_indices, 'Throughput'],
                    color='red', s=10, label='Filled Points')
        plt.title(f'Throughput with Imputation for link {link_name}')
        plt.xlabel('Timestamp')
        plt.ylabel('Throughput')
        plt.legend()
        plt.show()

        # Uncomment the following lines to save the plots as files
        # plot_filename = f'plot_{link_name}.png'
        # plt.savefig(plot_filename)
        # plt.close()

        print(f"Processed and plotted dataset for link: {link_name}")

# Example usage:
compare_and_plot_datasets('datasets/throughput/treated/cubic', 'datasets/dados-vazao-imputados/svd')


In [ ]:
df = pd.read_csv('datasets/throughput/treated/bbr/treated bbr esmond data ba-ap 07-08-2023.csv')

# Assuming your data is in a DataFrame df with columns 'Timestamp' and 'Throughput'
df['Throughput_Variance'] = df['Throughput'].rolling(window=10).var()

# Visualize the variance over time
import matplotlib.pyplot as plt

plt.plot(df['Timestamp'], df['Throughput_Variance'])
plt.xlabel('Timestamp')
plt.ylabel('Variance of Throughput')
plt.title('Throughput Variance Over Time')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load datasets and specify the datetime format explicitly
df = pd.read_csv('datasets/throughput/treated/cubic/treated cubic esmond data ba-ap 07-08-2023.csv')
df_interpolated = pd.read_csv('datasets/dados-vazao-imputados/svd/treated cubic esmond data ba-ap 07-08-2023.csv')

# Convert 'Timestamp' to datetime with specified format
df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%d-%m-%y %H:%M:%S', errors='coerce')
df_interpolated['Timestamp'] = pd.to_datetime(df_interpolated['Timestamp'], format='%d-%m-%y %H:%M:%S', errors='coerce')

# Remove rows with NaT (invalid date) in Timestamp, if any
df.dropna(subset=['Timestamp'], inplace=True)
df_interpolated.dropna(subset=['Timestamp'], inplace=True)

# Ensure 'Throughput' is numeric
df['Throughput'] = pd.to_numeric(df['Throughput'], errors='coerce')
df_interpolated['Throughput'] = pd.to_numeric(df_interpolated['Throughput'], errors='coerce')

# Set 'Timestamp' as the index for both datasets
df.set_index('Timestamp', inplace=True)
df_interpolated.set_index('Timestamp', inplace=True)

# Apply the `floor` method to create smaller time chunks (e.g., 3 hours)
df['Time_Chunk'] = df.index.to_series().dt.floor('3h')
df_interpolated['Time_Chunk'] = df_interpolated.index.to_series().dt.floor('3h')

# Calculate mean for each time chunk to confirm data exists in chunks
chunked_mean_original = df.groupby('Time_Chunk')['Throughput'].mean()
chunked_mean_interpolated = df_interpolated.groupby('Time_Chunk')['Throughput'].mean()

# Print the means to check if chunks contain data
print("Chunked Mean (Original):\n", chunked_mean_original)
print("Chunked Mean (Interpolated):\n", chunked_mean_interpolated)

# Plot chunked mean to analyze fluctuation by chunk, with increased width
plt.figure(figsize=(15, 6))  # Set figure size to 15 inches wide and 6 inches tall
plt.plot(chunked_mean_original.index, chunked_mean_original.values, label='Original Mean', marker='o')
plt.plot(chunked_mean_interpolated.index, chunked_mean_interpolated.values, label='Interpolated Mean', marker='o')
plt.xlabel('Time Chunk')
plt.ylabel('Mean Throughput in Each Chunk')
plt.title('Mean Throughput Across Different Time Chunks: Original vs Interpolated')
plt.legend()
plt.show()


In [ ]:
# Split data into chunks based on time intervals (e.g., every 1 hour if timestamp is datetime)
df['Time_Chunk'] = pd.to_datetime(df['Timestamp']).dt.floor('6h')
chunked_variance = df.groupby('Time_Chunk')['Throughput'].var()

# Plot chunked variance to analyze fluctuation by chunk
plt.plot(chunked_variance.index, chunked_variance.values)
plt.xlabel('Time Chunk')
plt.ylabel('Variance in Each Chunk')
plt.title('Variance Across Different Time Chunks')
plt.show()


In [ ]:
import os
import pandas as pd

# Diretório onde estão os arquivos CSV
diretorio = '../datasets/vazao/10-06-2023/tratado/bbr/'

# Percorra todos os arquivos na pasta
for arquivo in os.listdir(diretorio):
    if arquivo.endswith('.csv') and arquivo.startswith('tratado bbr esmond'):
        caminho_arquivo = os.path.join(diretorio, arquivo)
        df = pd.read_csv(caminho_arquivo)

        longest_interval = []
        current_interval = []
        for index, row in df.iterrows():
            if row['Vazao'] != -1:
                current_interval.append(row)
            else:
                if len(current_interval) > len(longest_interval):
                    longest_interval = current_interval
                current_interval = []

        # Verifique novamente no final, caso o intervalo mais longo termine no final do arquivo
        if len(current_interval) > len(longest_interval):
            longest_interval = current_interval

        diretorio_salvar = '../datasets/intervalos-completos/vazao/bbr 06-10-2023/'
        if len(longest_interval) >= 10:
            longest_df = pd.DataFrame(longest_interval)
            output_file = os.path.splitext(arquivo)[0] + '_longest_interval.csv'
            longest_df.to_csv(os.path.join(diretorio_salvar, output_file), index=False)


In [ ]:
def decomposicao_svd(df):
    U, S, Vt = np.linalg.svd(df, full_matrices=True)
    return U, S, Vt

def grafico_variabilidade(variabilidade, S):
    plt.plot(range(1, len(variabilidade) + 1), variabilidade, marker='o', markersize=1, markerfacecolor='teal', markeredgecolor='teal', color='darkturquoise')
    plt.xlabel('Número de Valores Singulares')
    plt.ylabel('Variabilidade Acumulada')
    plt.title('Valores Singulares por Variabilidade Acumulada')
    plt.grid(color='lightgray', alpha=0.7)
    plt.show()

def componentes_principais(r, U, S, Vt):
    U_reduced = U[:, :r]
    S_reduced = S[:r]
    Vt_reduced = Vt[:r, :]
    return U_reduced, S_reduced, Vt_reduced
#trasnforma a matriz de volta em um dataframe
def matriztodf(dataframes):
    df = pd.DataFrame(dataframes)
    q = df.shape[0]*df.shape[1]
    df = df.transpose()
    df = df.to_numpy().reshape(-1, q)
    df = df.transpose()
    df = pd.DataFrame(df)
    if 0 in df.columns:
        df = df.rename(columns={0: 'Throughput'})
    return df

#calcula rmse dos valores gerados no svd com os valores 'originais'
def calcular_rmse(df1, df2, coluna):
    indices_comuns = df1.index.intersection(df2.index)
    valores_df1 = df1.loc[indices_comuns, coluna]
    valores_df2 = df2.loc[indices_comuns, coluna]
    rmse = np.sqrt(np.mean((valores_df1 - valores_df2) ** 2))
    return rmse

#gera arquivo csv final com a imputacao
def gerar_arq_csv(df1, df2, caminho_base, nome_arquivo_csv):
    df1['Throughput'] = pd.NA
    df1['Throughput'] = df2['Throughput']
    #exclui as ultimas linhas do arquivo que nao foi feita imputacao
    df1 = df1.dropna(subset=['Throughput'])
    caminho_svd = os.path.join(caminho_base, 'svd')
    if not os.path.exists(caminho_svd):
        os.makedirs(caminho_svd)
    caminho_arquivo_csv = os.path.join(caminho_svd, nome_arquivo_csv)
    # Salva o DataFrame resultante em um arquivo CSV
    df1.to_csv(caminho_arquivo_csv, index=False)
    print(f"Arquivo CSV '{caminho_arquivo_csv}' gerado com sucesso!")
    return df1

In [ ]:
#função para tratamento inicial do csv:
#corrigir aqui pq algo ta fazendo bugar o rj-ap

def decomposicao_svd(df):
    U, S, Vt = np.linalg.svd(df, full_matrices=True)
    return U, S, Vt

def grafico_variabilidade(variabilidade, S):
    plt.plot(range(1, len(variabilidade) + 1), variabilidade, marker='o', markersize=1, markerfacecolor='teal', markeredgecolor='teal', color='darkturquoise')
    plt.xlabel('Número de Valores Singulares')
    plt.ylabel('Variabilidade Acumulada')
    plt.title('Valores Singulares por Variabilidade Acumulada')
    plt.grid(color='lightgray', alpha=0.7)
    plt.show()

def componentes_principais(r, U, S, Vt):
    U_reduced = U[:, :r]
    S_reduced = S[:r]
    Vt_reduced = Vt[:r, :]
    return U_reduced, S_reduced, Vt_reduced
#trasnforma a matriz de volta em um dataframe
def matriztodf(dataframes):
    df = pd.DataFrame(dataframes)
    q = df.shape[0]*df.shape[1]
    df = df.transpose()
    df = df.to_numpy().reshape(-1, q)
    df = df.transpose()
    df = pd.DataFrame(df)
    if 0 in df.columns:
        df = df.rename(columns={0: 'Throughput'})
    return df

#calcula rmse dos valores gerados no svd com os valores 'originais'
def calcular_rmse(df1, df2, coluna):
    indices_comuns = df1.index.intersection(df2.index)
    valores_df1 = df1.loc[indices_comuns, coluna]
    valores_df2 = df2.loc[indices_comuns, coluna]
    rmse = np.sqrt(np.mean((valores_df1 - valores_df2) ** 2))
    return rmse

#gera arquivo csv final com a imputacao
def gerar_arq_csv(df1, df2, caminho_base, nome_arquivo_csv):
    df1['Throughput'] = pd.NA
    df1['Throughput'] = df2['Throughput']
    #exclui as ultimas linhas do arquivo que nao foi feita imputacao
    df1 = df1.dropna(subset=['Throughput'])
    caminho_svd = os.path.join(caminho_base, 'svd')
    if not os.path.exists(caminho_svd):
        os.makedirs(caminho_svd)
    caminho_arquivo_csv = os.path.join(caminho_svd, nome_arquivo_csv)
    # Salva o DataFrame resultante em um arquivo CSV
    df1.to_csv(caminho_arquivo_csv, index=False)
    print(f"Arquivo CSV '{caminho_arquivo_csv}' gerado com sucesso!")
    return df1

def outlier_removal(df, column):

    df[column] = df[column].replace(-1, np.nan)

    r = df[column].dropna().to_numpy()
    
    if r.size == 0:
        print("Coluna não contém valores suficientes para análise.")
        return df

    r_max = np.max(r) 
    r = r / r_max  

    perc_min = []
    p_min = np.linspace(0.1, 2, 20)
    for i in p_min:
        perc_min.append(np.percentile(r, i))
    diff_perc_min = np.diff(perc_min)
    index_min = np.argmax(diff_perc_min)  
    thres_min = np.mean(perc_min[index_min:index_min + 2])

    perc_max = []
    p_max = np.linspace(98, 100, 20)
    for i in p_max:
        perc_max.append(np.percentile(r, i))
    diff_perc_max = np.diff(perc_max)
    index_max = np.argmax(diff_perc_max)  
    thres_max = np.mean(perc_max[index_max:index_max + 2])

    r_filtered = np.where((r < thres_min) | (r > thres_max), np.nan, r)

    r_filtered = r_filtered * r_max  

    df_filtered = df.copy()
    df_filtered.loc[~df[column].isna(), column] = r_filtered

    return df_filtered

def matriz(path):
    df = pd.read_csv(path)
    df = outlier_removal(df, 'Throughput')
    df_datetime = df.copy()
    df_datetime.drop(columns=['Throughput'], inplace = True)
    df['Throughput'] = df['Throughput'].replace(-1, np.nan)
    
    Throughput = df['Throughput'].values
    num_dados = len(Throughput)
    num_colunas = num_dados // 28
    matriz = Throughput[:num_colunas*28].reshape(num_colunas, 28).T
    matriz_original = pd.DataFrame(matriz)
    df_interpolado = df["Throughput"].interpolate(method='linear', limit_direction='both')
    Throughput_=df_interpolado.values
    matriz_interpolado = Throughput_[:num_colunas*28].reshape(num_colunas, 28).T
    matriz_interpolado= pd.DataFrame(matriz_interpolado)
    mask = np.isnan(matriz_original.values)
    matriz_mascara = pd.DataFrame(mask)
    return matriz_original, matriz_mascara, matriz_interpolado, df_datetime#, r_max

In [ ]:
caminhos_csv = ['datasets/throughput/treated/cubic/treated cubic esmond data ap-ba 07-03-2023.csv', 'datasets/throughput/treated/cubic/treated cubic esmond data ba-ap 07-08-2023.csv', 'datasets/throughput/treated/cubic/treated cubic esmond data ba-rj 07-03-2023.csv','datasets/throughput/treated/cubic/treated cubic esmond data es-ap 07-08-2023.csv', 'datasets/throughput/treated/cubic/treated cubic esmond data es-rs 07-03-2023.csv']

In [ ]:
def get_dataset_path_list(diretory):
    dataset_path_list = []
    for file in os.listdir(diretory):
        path = os.path.join(diretory, file)
        dataset_path_list.append(path)
    return dataset_path_list

In [ ]:
caminhos_csv = get_dataset_path_list('datasets/treated_longest_interval_with_nans/')
resultados = {}


for caminho_csv in caminhos_csv:
    nome_arquivo = os.path.basename(caminho_csv)
    
    resultados[nome_arquivo] = {'interpolacao_linear': None, 'svd_final': None}
    df_matriz, df_mask, df_interpolado, df_datetime = matriz(caminho_csv)
    resultados[nome_arquivo]['interpolacao_linear'] = df_interpolado.copy()

    A_anterior = df_interpolado.values.copy()
    rmse = float('inf') 
    max_iter = 300
    n_iter = 0


    while rmse >= 1e-3 and n_iter<=max_iter:  
        U, S, Vt = decomposicao_svd(df_interpolado)
        variabilidade = np.cumsum(S**2) / np.sum(S**2)

        porcentagem_variabilidade = 0.95
        r = np.where(variabilidade >= porcentagem_variabilidade)[0][0] + 1
        
        #print(f'Número de valores singulares para atingir {porcentagem_variabilidade*100}% de variabilidade: {r}')

        U_reduzido, S_reduzido, Vt_reduzido = componentes_principais(r, U, S, Vt)
        S_reduzido_matriz = np.diag(S_reduzido)

        A_aproximada = np.dot(np.dot(U_reduzido, S_reduzido_matriz), Vt_reduzido)
        A_aproximada_df = pd.DataFrame(A_aproximada)

        df_matriz_preenchida = df_matriz.fillna(A_aproximada_df)
        
        resultados[nome_arquivo]['svd_final'] = df_matriz_preenchida

        # Atualiza df_interpolado para a próxima iteração
        df_interpolado = df_matriz_preenchida.values
        
        # Calcular o RMSE entre a matriz atual e a anterior
        rmse = np.sqrt(np.mean((A_aproximada - A_anterior) ** 2))

        # Atualiza A_anterior para a próxima comparação
        A_anterior = A_aproximada.copy()

        n_iter +=1

    print(f'RMSE na iteração atual: {rmse}')
    print(f'Finalizando processamento para {caminho_csv}')

    svd = matriztodf(resultados[nome_arquivo]["svd_final"])
    interpolacao = matriztodf(resultados[nome_arquivo]["interpolacao_linear"])
    mask = matriztodf(df_mask)
    dfs_reshaped = []
    dfs_reshaped.append(interpolacao) #0
    dfs_reshaped.append(svd) #1
    dfs_reshaped.append(mask) #2
    # Chama a função para plotar os dados
    # plot_imputed_data(dfs_reshaped)
    base_path = 'datasets/throughput/imputed-treated-longest-interval'
    gerar_arq_csv(df_datetime, dfs_reshaped[1], base_path, nome_arquivo)


In [ ]:
def impute_knn(df, k=5):
    imputer = KNNImputer(n_neighbors=k)
    df['Throughput'] = imputer.fit_transform(df[['Throughput']])
    return df

def impute_rolling_median(df, window_size=3):
    df['Throughput'] = df['Throughput'].fillna(df['Throughput'].rolling(window=window_size, min_periods=1).median())
    return df

def impute_rolling_average(df, window_size=3):
    df['Throughput'] = df['Throughput'].fillna(df['Throughput'].rolling(window=window_size, min_periods=1).mean())
    return df

In [ ]:
resultados = {}

for caminho in caminhos_csv:
    # Carregar o arquivo
    df = pd.read_csv(caminho)

    df = outlier_removal(df, 'Throughput')
    
    # Aplicar imputação KNN
    df_knn = impute_knn(df.copy())
    
    # Aplicar mediana móvel
    df_rolling_median = impute_rolling_median(df.copy())
    
    # Aplicar média móvel
    df_rolling_average = impute_rolling_average(df.copy())

    df_interpolation = df.interpolate(method='linear', limit_direction='both')
    
    # Salvar os resultados em novos arquivos
    output_knn = caminho.replace('datasets/treated_longest_interval_with_nans/', 'datasets/throughput/imputed-treated-longest-interval/knn/')
    output_median = caminho.replace('datasets/treated_longest_interval_with_nans/', 'datasets/throughput/imputed-treated-longest-interval/mediana-movel/')
    output_average = caminho.replace('datasets/treated_longest_interval_with_nans/', 'datasets/throughput/imputed-treated-longest-interval/media-movel/')
    output_interpolation = caminho.replace('datasets/treated_longest_interval_with_nans/', 'datasets/throughput/imputed-treated-longest-interval/interpolacao-linear/')
    
    df_knn.to_csv(output_knn, index=False)
    df_rolling_median.to_csv(output_median, index=False)
    df_rolling_average.to_csv(output_average, index=False)
    df_interpolation.to_csv(output_interpolation, index=False)

    print(f"Processed file: {caminho}")

In [ ]:
import os
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt

# Directories
diretorio1 = 'datasets/treated_longest_interval_with_nans/'
diretorio2 = 'datasets/throughput/imputed-treated-longest-interval/'
tecnicas = ['knn', 'media-movel', 'mediana-movel', 'svd', 'interpolacao-linear']

# Initialize a dictionary to store RMSE results
rmse_results = {}

# Loop over each file in the first directory
for arquivo in os.listdir(diretorio1):
    path = os.path.join(diretorio1, arquivo)
    df = pd.read_csv(path)
    
    # Initialize a dictionary for this file
    rmse_results[arquivo] = {}
    
    # Loop over each technique
    for tecnica in tecnicas:
        path2 = os.path.join(diretorio2, tecnica, arquivo)
        
        # Check if the file exists to avoid errors
        if not os.path.exists(path2):
            continue
        
        df2 = pd.read_csv(path2)
        
        # Ensure both dataframes have the same columns
        common_columns = df.columns.intersection(df2.columns)
        df = df[common_columns]
        df2 = df2[common_columns]
        
        # Compute RMSE between df and df2 for numeric columns
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        diff = df[numeric_cols] - df2[numeric_cols]
        mse = (diff ** 2).mean()
        rmse = np.sqrt(mse)
        
        # Convert RMSE Series to a dictionary for JSON serialization
        rmse_dict = rmse.to_dict()
        
        # Store the RMSE for this technique
        rmse_results[arquivo][tecnica] = rmse_dict

# Save the RMSE results to a JSON file
with open('rmse_results.json', 'w') as f:
    json.dump(rmse_results, f, indent=4)




In [ ]:
def data_imputation_datasets(datasets_path_list, imputation_column='Throughput'):
    list_imputed_dfs = []
    for path_dataset in datasets_path_list: 
        original_df = pd.read_csv(path_dataset)
        df = original_df.copy()

        # Removing outliers for datasets that are being imputed with knn, linear interp and moving average and median
        df = outlier_removal(df, imputation_column) 

        df_imputed_linear_interpolation = df.copy().interpolate(method='linear', limit_direction='both')

        list_imputed_dfs.append(df_imputed_linear_interpolation)

        df_imputed_knn = impute_knn(df.copy(), imputation_column=imputation_column)

        list_imputed_dfs.append(df_imputed_knn)

        df_imputed_rolling_average = impute_rolling_average(df.copy(), imputation_column=imputation_column)

        list_imputed_dfs.append(df_imputed_rolling_average)

        df_imputed_rolling_median = impute_rolling_median(df.copy(), imputation_column=imputation_column)

        list_imputed_dfs.append(df_imputed_rolling_median)

        # For specified pre processing for svd, the path is given, not the already without outlier df
        df_imputed_svd = impute_svd(path_dataset)

        list_imputed_dfs.append(df_imputed_svd)

        return list_imputed_dfs

def impute_knn(df, imputation_column, k=5):
    imputer = KNNImputer(n_neighbors=k)
    df[imputation_column] = imputer.fit_transform(df[[imputation_column]])
    return df

def impute_rolling_median(df, imputation_column, window_size=3):
    df[imputation_column] = df[imputation_column].fillna(df[imputation_column].rolling(window=window_size, min_periods=1).median())
    return df

def impute_rolling_average(df, imputation_column, window_size=3):
    df[imputation_column] = df[imputation_column].fillna(df[imputation_column].rolling(window=window_size, min_periods=1).mean())
    return df 

def outlier_removal(df, column):

    df[column] = df[column].replace(-1, np.nan)

    r = df[column].dropna().to_numpy()
    
    if r.size == 0:
        print("Column does not contain enough values ​​for analysis.")
        return df

    r_max = np.max(r) 
    r = r / r_max  

    perc_min = []
    p_min = np.linspace(0.1, 2, 20)
    for i in p_min:
        perc_min.append(np.percentile(r, i))
    diff_perc_min = np.diff(perc_min)
    index_min = np.argmax(diff_perc_min)  
    thres_min = np.mean(perc_min[index_min:index_min + 2])

    perc_max = []
    p_max = np.linspace(98, 100, 20)
    for i in p_max:
        perc_max.append(np.percentile(r, i))
    diff_perc_max = np.diff(perc_max)
    index_max = np.argmax(diff_perc_max)  
    thres_max = np.mean(perc_max[index_max:index_max + 2])

    r_filtered = np.where((r < thres_min) | (r > thres_max), np.nan, r)

    r_filtered = r_filtered * r_max  

    df_filtered = df.copy()
    df_filtered.loc[~df[column].isna(), column] = r_filtered

    return df_filtered


def impute_svd(path_dataset):
    def decomposicao_svd(df):
        U, S, Vt = np.linalg.svd(df, full_matrices=True)
        return U, S, Vt

    # def grafico_variabilidade(variabilidade, S):
    #     plt.plot(range(1, len(variabilidade) + 1), variabilidade, marker='o', markersize=1, markerfacecolor='teal', markeredgecolor='teal', color='darkturquoise')
    #     plt.xlabel('Número de Valores Singulares')
    #     plt.ylabel('Variabilidade Acumulada')
    #     plt.title('Valores Singulares por Variabilidade Acumulada')
    #     plt.grid(color='lightgray', alpha=0.7)
    #     plt.show()

    def componentes_principais(r, U, S, Vt):
        U_reduced = U[:, :r]
        S_reduced = S[:r]
        Vt_reduced = Vt[:r, :]
        return U_reduced, S_reduced, Vt_reduced
    #trasnforma a matriz de volta em um dataframe
    def matriztodf(dataframes):
        df = pd.DataFrame(dataframes)
        q = df.shape[0]*df.shape[1]
        df = df.transpose()
        df = df.to_numpy().reshape(-1, q)
        df = df.transpose()
        df = pd.DataFrame(df)
        if 0 in df.columns:
            df = df.rename(columns={0: 'Throughput'})
        return df

    #calcula rmse dos valores gerados no svd com os valores 'originais'
    # def calcular_rmse(df1, df2, coluna):
    #     indices_comuns = df1.index.intersection(df2.index)
    #     valores_df1 = df1.loc[indices_comuns, coluna]
    #     valores_df2 = df2.loc[indices_comuns, coluna]
    #     rmse = np.sqrt(np.mean((valores_df1 - valores_df2) ** 2))
    #     return rmse

    #gera arquivo csv final com a imputacao
    def gerar_arq_csv(df1, df2, caminho_base, nome_arquivo_csv):
        df1['Throughput'] = pd.NA
        df1['Throughput'] = df2['Throughput']
        #exclui as ultimas linhas do arquivo que nao foi feita imputacao
        df1 = df1.dropna(subset=['Throughput'])
        caminho_svd = os.path.join(caminho_base, 'svd')
        if not os.path.exists(caminho_svd):
            os.makedirs(caminho_svd)
        caminho_arquivo_csv = os.path.join(caminho_svd, nome_arquivo_csv)
        # Salva o DataFrame resultante em um arquivo CSV
        df1.to_csv(caminho_arquivo_csv, index=False)
        print(f"Arquivo CSV '{caminho_arquivo_csv}' gerado com sucesso!")
        return df1

    def matriz(path):
        df = pd.read_csv(path)
        df = outlier_removal(df, 'Throughput')
        df_datetime = df.copy()
        df_datetime.drop(columns=['Throughput'], inplace = True)
        df['Throughput'] = df['Throughput'].replace(-1, np.nan)
        
        Throughput = df['Throughput'].values
        num_dados = len(Throughput)
        num_colunas = num_dados // 28
        matriz = Throughput[:num_colunas*28].reshape(num_colunas, 28).T
        matriz_original = pd.DataFrame(matriz)
        df_interpolado = df["Throughput"].interpolate(method='linear', limit_direction='both')
        Throughput_=df_interpolado.values
        matriz_interpolado = Throughput_[:num_colunas*28].reshape(num_colunas, 28).T
        matriz_interpolado= pd.DataFrame(matriz_interpolado)
        mask = np.isnan(matriz_original.values)
        matriz_mascara = pd.DataFrame(mask)
        return matriz_original, matriz_mascara, matriz_interpolado, df_datetime#, r_max
    
    nome_arquivo = os.path.basename(path_dataset)
    
    resultados[nome_arquivo] = {'interpolacao_linear': None, 'svd_final': None}
    df_matriz, df_mask, df_interpolado, df_datetime = matriz(path_dataset)
    resultados[nome_arquivo]['interpolacao_linear'] = df_interpolado.copy()

    A_anterior = df_interpolado.values.copy()
    rmse = float('inf') 
    max_iter = 300
    n_iter = 0

    while rmse >= 1e-3 and n_iter<=max_iter:  
        U, S, Vt = decomposicao_svd(df_interpolado)
        variabilidade = np.cumsum(S**2) / np.sum(S**2)

        porcentagem_variabilidade = 0.95

        if np.any(variabilidade >= porcentagem_variabilidade):
            r = np.where(variabilidade >= porcentagem_variabilidade)[0][0] + 1
        else:
            # Set a default for `r` if threshold isn't met
            r = len(S)  # or choose another fallback value like `r = 1`
        
        #print(f'Número de valores singulares para atingir {porcentagem_variabilidade*100}% de variabilidade: {r}')

        U_reduzido, S_reduzido, Vt_reduzido = componentes_principais(r, U, S, Vt)
        S_reduzido_matriz = np.diag(S_reduzido)

        A_aproximada = np.dot(np.dot(U_reduzido, S_reduzido_matriz), Vt_reduzido)
        A_aproximada_df = pd.DataFrame(A_aproximada)

        df_matriz_preenchida = df_matriz.fillna(A_aproximada_df)
        
        resultados[nome_arquivo]['svd_final'] = df_matriz_preenchida

        # Atualiza df_interpolado para a próxima iteração
        df_interpolado = df_matriz_preenchida.values
        
        # Calcular o RMSE entre a matriz atual e a anterior
        rmse = np.sqrt(np.mean((A_aproximada - A_anterior) ** 2))

        # Atualiza A_anterior para a próxima comparação
        A_anterior = A_aproximada.copy()

        n_iter +=1

    # print(f'RMSE na iteração atual: {rmse}')
    # print(f'Finalizando processamento para {caminho_csv}')

    svd = matriztodf(resultados[nome_arquivo]["svd_final"])
    # interpolacao = matriztodf(resultados[nome_arquivo]["interpolacao_linear"])
    # mask = matriztodf(df_mask)
    # dfs_reshaped = []
    # dfs_reshaped.append(interpolacao) #0
    # dfs_reshaped.append(svd) #1
    # dfs_reshaped.append(mask) #2
    # Chama a função para plotar os dados
    # plot_imputed_data(dfs_reshaped)
    # base_path = 'datasets/dados-vazao-imputados/svd'
    # gerar_arq_csv(df_datetime, dfs_reshaped[1], base_path, nome_arquivo)
    return svd

In [ ]:
def get_dataset_path_list(diretory):
    dataset_path_list = []
    for file in os.listdir(diretory):
        path = os.path.join(diretory, file)
        dataset_path_list.append(path)
    return dataset_path_list

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

# Load the JSON data from the file
with open('rmse_results.json', 'r') as f:
    json_data = json.load(f)

# Function to parse the protocol and link from the file name
def parse_file_name(file_name):
    parts = file_name.split()
    protocol = parts[1]  # Second word is the protocol
    try:
        link_index = parts.index('data') + 1
        link = parts[link_index]
    except ValueError:
        link = 'Unknown'
    return protocol, link

# Function to load and process the data
def load_data(json_data):
    data_list = []
    for file_name, techniques in json_data.items():
        protocol, link = parse_file_name(file_name)
        for imputation, metrics in techniques.items():
            throughput = metrics['Throughput']
            data_list.append({
                'File': file_name,
                'Protocol': protocol,
                'Link': link,
                'Imputation': imputation,
                'Throughput': throughput
            })
    return data_list

# Function to plot the data
def plot_data(data_list):
    df = pd.DataFrame(data_list)
    # Create a shorter label for plotting
    df['Label'] = df['Protocol'] + ' ' + df['Link']

    # Pivot the DataFrame to have 'Label' as index and 'Imputation' as columns
    pivot_df = df.pivot(index='Label', columns='Imputation', values='Throughput')

    # Plot the data
    pivot_df.plot(kind='bar', figsize=(14, 8))
    plt.xlabel('Protocol and Link')
    plt.ylabel('Throughput')
    plt.title('Throughput by Imputation Technique, Protocol, and Link')
    plt.legend(title='Imputation Technique', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

# Load and process the data
data_list = load_data(json_data)

# Plot the data
plot_data(data_list)


In [ ]:
def data_imputation_datasets(datasets_path, saving_path, imputation_column='Throughput'):
    list_imputed_dfs = []
    for path in os.listdir(datasets_path):
        path_dataset = os.path.join(datasets_path, path)
        original_df = pd.read_csv(path_dataset)
        df = original_df.copy()

        # Removing outliers for datasets that are being imputed with knn, linear interp and moving average and median
        df = outlier_removal(df, imputation_column) 

        df_imputed_linear_interpolation = df.copy().interpolate(method='linear', limit_direction='both')

        list_imputed_dfs.append(df_imputed_linear_interpolation)

        df_imputed_knn = impute_knn(df.copy(), imputation_column=imputation_column)

        list_imputed_dfs.append(df_imputed_knn)

        df_imputed_rolling_average = impute_rolling_average(df.copy(), imputation_column=imputation_column)

        list_imputed_dfs.append(df_imputed_rolling_average)

        df_imputed_rolling_median = impute_rolling_median(df.copy(), imputation_column=imputation_column)

        list_imputed_dfs.append(df_imputed_rolling_median)

        # For specified pre processing for svd, the path is given, not the already without outlier df
        df_imputed_svd = impute_svd(path_dataset)

        list_imputed_dfs.append(df_imputed_svd)

        if not os.path.exists(f'{saving_path}/knn/'):
            os.makedirs(f'{saving_path}/knn/')
        if not os.path.exists(f'{saving_path}/interpolacao-linear/'):
            os.makedirs(f'{saving_path}/interpolacao-linear/')
        if not os.path.exists(f'{saving_path}/mediana-movel/'):
            os.makedirs(f'{saving_path}/mediana-movel/')
        if not os.path.exists(f'{saving_path}/media-movel/'):
            os.makedirs(f'{saving_path}/media-movel/')
        if not os.path.exists(f'{saving_path}/svd/'):
            os.makedirs(f'{saving_path}/svd/')

        df_imputed_knn.to_csv(f'{saving_path}/knn/{path}')
        df_imputed_linear_interpolation.to_csv(f'{saving_path}/interpolacao-linear/{path}')
        df_imputed_rolling_median.to_csv(f'{saving_path}/mediana-movel/{path}')
        df_imputed_rolling_average.to_csv(f'{saving_path}/media-movel/{path}')
        df_imputed_svd.to_csv(f'{saving_path}/svd/{path}')

    return list_imputed_dfs

def impute_knn(df, imputation_column, k=5):
    imputer = KNNImputer(n_neighbors=k)
    df[imputation_column] = imputer.fit_transform(df[[imputation_column]])
    return df

def impute_rolling_median(df, imputation_column, window_size=3):
    df[imputation_column] = df[imputation_column].fillna(df[imputation_column].rolling(window=window_size, min_periods=1).median())
    return df

def impute_rolling_average(df, imputation_column, window_size=3):
    df[imputation_column] = df[imputation_column].fillna(df[imputation_column].rolling(window=window_size, min_periods=1).mean())
    return df 

def outlier_removal(df, column):

    df[column] = df[column].replace(-1, np.nan)

    r = df[column].dropna().to_numpy()
    
    if r.size == 0:
        print("Column does not contain enough values ​​for analysis.")
        return df

    r_max = np.max(r) 
    r = r / r_max  

    perc_min = []
    p_min = np.linspace(0.1, 2, 20)
    for i in p_min:
        perc_min.append(np.percentile(r, i))
    diff_perc_min = np.diff(perc_min)
    index_min = np.argmax(diff_perc_min)  
    thres_min = np.mean(perc_min[index_min:index_min + 2])

    perc_max = []
    p_max = np.linspace(98, 100, 20)
    for i in p_max:
        perc_max.append(np.percentile(r, i))
    diff_perc_max = np.diff(perc_max)
    index_max = np.argmax(diff_perc_max)  
    thres_max = np.mean(perc_max[index_max:index_max + 2])

    r_filtered = np.where((r < thres_min) | (r > thres_max), np.nan, r)

    r_filtered = r_filtered * r_max  

    df_filtered = df.copy()
    df_filtered.loc[~df[column].isna(), column] = r_filtered

    return df_filtered


def impute_svd(path_dataset):
    def decomposicao_svd(df):
        U, S, Vt = np.linalg.svd(df, full_matrices=True)
        return U, S, Vt

    # def grafico_variabilidade(variabilidade, S):
    #     plt.plot(range(1, len(variabilidade) + 1), variabilidade, marker='o', markersize=1, markerfacecolor='teal', markeredgecolor='teal', color='darkturquoise')
    #     plt.xlabel('Número de Valores Singulares')
    #     plt.ylabel('Variabilidade Acumulada')
    #     plt.title('Valores Singulares por Variabilidade Acumulada')
    #     plt.grid(color='lightgray', alpha=0.7)
    #     plt.show()

    def componentes_principais(r, U, S, Vt):
        U_reduced = U[:, :r]
        S_reduced = S[:r]
        Vt_reduced = Vt[:r, :]
        return U_reduced, S_reduced, Vt_reduced
    #trasnforma a matriz de volta em um dataframe
    def matriztodf(dataframes):
        df = pd.DataFrame(dataframes)
        q = df.shape[0]*df.shape[1]
        df = df.transpose()
        df = df.to_numpy().reshape(-1, q)
        df = df.transpose()
        df = pd.DataFrame(df)
        if 0 in df.columns:
            df = df.rename(columns={0: 'Throughput'})
        return df

    #calcula rmse dos valores gerados no svd com os valores 'originais'
    # def calcular_rmse(df1, df2, coluna):
    #     indices_comuns = df1.index.intersection(df2.index)
    #     valores_df1 = df1.loc[indices_comuns, coluna]
    #     valores_df2 = df2.loc[indices_comuns, coluna]
    #     rmse = np.sqrt(np.mean((valores_df1 - valores_df2) ** 2))
    #     return rmse

    #gera arquivo csv final com a imputacao
    def gerar_arq_csv(df1, df2, caminho_base, nome_arquivo_csv):
        df1['Throughput'] = pd.NA
        df1['Throughput'] = df2['Throughput']
        #exclui as ultimas linhas do arquivo que nao foi feita imputacao
        df1 = df1.dropna(subset=['Throughput'])
        caminho_svd = os.path.join(caminho_base, 'svd')
        if not os.path.exists(caminho_svd):
            os.makedirs(caminho_svd)
        caminho_arquivo_csv = os.path.join(caminho_svd, nome_arquivo_csv)
        # Salva o DataFrame resultante em um arquivo CSV
        df1.to_csv(caminho_arquivo_csv, index=False)
        print(f"Arquivo CSV '{caminho_arquivo_csv}' gerado com sucesso!")
        return df1

    def matriz(path):
        df = pd.read_csv(path)
        df = outlier_removal(df, 'Throughput')
        df_datetime = df.copy()
        df_datetime.drop(columns=['Throughput'], inplace = True)
        df['Throughput'] = df['Throughput'].replace(-1, np.nan)
        
        Throughput = df['Throughput'].values
        num_dados = len(Throughput)
        num_colunas = num_dados // 28
        matriz = Throughput[:num_colunas*28].reshape(num_colunas, 28).T
        matriz_original = pd.DataFrame(matriz)
        df_interpolado = df["Throughput"].interpolate(method='linear', limit_direction='both')
        Throughput_=df_interpolado.values
        matriz_interpolado = Throughput_[:num_colunas*28].reshape(num_colunas, 28).T
        matriz_interpolado= pd.DataFrame(matriz_interpolado)
        mask = np.isnan(matriz_original.values)
        matriz_mascara = pd.DataFrame(mask)
        return matriz_original, matriz_mascara, matriz_interpolado, df_datetime#, r_max
    
    resultados = {}
    nome_arquivo = os.path.basename(path_dataset)
    
    resultados[nome_arquivo] = {'interpolacao_linear': None, 'svd_final': None}
    df_matriz, df_mask, df_interpolado, df_datetime = matriz(path_dataset)
    resultados[nome_arquivo]['interpolacao_linear'] = df_interpolado.copy()

    A_anterior = df_interpolado.values.copy()
    rmse = float('inf') 
    max_iter = 300
    n_iter = 0

    while rmse >= 1e-3 and n_iter<=max_iter:  
        U, S, Vt = decomposicao_svd(df_interpolado)
        variabilidade = np.cumsum(S**2) / np.sum(S**2)

        porcentagem_variabilidade = 0.95

        if np.any(variabilidade >= porcentagem_variabilidade):
            r = np.where(variabilidade >= porcentagem_variabilidade)[0][0] + 1
        else:
            break
        
        #print(f'Número de valores singulares para atingir {porcentagem_variabilidade*100}% de variabilidade: {r}')

        U_reduzido, S_reduzido, Vt_reduzido = componentes_principais(r, U, S, Vt)
        S_reduzido_matriz = np.diag(S_reduzido)

        A_aproximada = np.dot(np.dot(U_reduzido, S_reduzido_matriz), Vt_reduzido)
        A_aproximada_df = pd.DataFrame(A_aproximada)

        df_matriz_preenchida = df_matriz.fillna(A_aproximada_df)
        
        resultados[nome_arquivo]['svd_final'] = df_matriz_preenchida

        # Atualiza df_interpolado para a próxima iteração
        df_interpolado = df_matriz_preenchida.values
        
        # Calcular o RMSE entre a matriz atual e a anterior
        rmse = np.sqrt(np.mean((A_aproximada - A_anterior) ** 2))

        # Atualiza A_anterior para a próxima comparação
        A_anterior = A_aproximada.copy()

        n_iter +=1

    # print(f'RMSE na iteração atual: {rmse}')
    # print(f'Finalizando processamento para {caminho_csv}')

    svd = matriztodf(resultados[nome_arquivo]["svd_final"])
    # interpolacao = matriztodf(resultados[nome_arquivo]["interpolacao_linear"])
    # mask = matriztodf(df_mask)
    # dfs_reshaped = []
    # dfs_reshaped.append(interpolacao) #0
    # dfs_reshaped.append(svd) #1
    # dfs_reshaped.append(mask) #2
    # Chama a função para plotar os dados
    # plot_imputed_data(dfs_reshaped)
    # base_path = 'datasets/dados-vazao-imputados/svd'
    # gerar_arq_csv(df_datetime, dfs_reshaped[1], base_path, nome_arquivo)
    return svd

In [ ]:
datasets = '../datasets/treated longest interval with failures/'

saving_imputation_datasets = '../datasets/imputed treated longest interval'

In [ ]:
data_imputation_datasets(datasets, saving_imputation_datasets)